# trap_sim_nullspace Workflow Notebook

This notebook runs the full pipeline described in the README with minimal changes to the existing codebase.

Run cells from top to bottom.

Pipeline stages:
1. Pre-process chip GDS
2. Generate journal file for meshing/export
3. Generate grid file
4. Generate input deck
5. Run nullspace simulation (external command)
6. Post-process simulation output

In [ ]:
from pathlib import Path
import os
import sys

# Ensure we can import from the project root when notebook lives in lionix_gate/
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Current working directory:", Path.cwd().resolve())

In [ ]:
# Step 1: Pre-process GDS
from pre_process import run_pre_process

preprocessed_gds = run_pre_process(
    design_file="output/lionix_gate/chip_design.gds",
    output_file="output/lionix_gate/chip_design_merged_flattened.gds",
)
preprocessed_gds

In [ ]:
# Step 2: Generate journal file
from journal_file import generate_journal

journal_file = generate_journal(
    processed_gds=preprocessed_gds,
    journal_filename="output/lionix_gate/chip_geometry.jou",
    abaqus_filename="run_file/chip_geometry.inp",
    interactive_plot=False,
)
journal_file

In [ ]:
# Step 3: Generate grid file
import numpy as np
from grid_file_generation import generate_grid_file

grid_file = generate_grid_file(
    x_scan=[0],
    y_scan=[0],
    z_scan=np.linspace(45, 55, 101),
    filename="z_scan.h5",
)
grid_file

In [ ]:
# Step 4: Generate input deck
from input_deck_generation import generate_input_deck

input_deck = generate_input_deck(
    mesh_file="chip.inp",
    input_deck_filename="rf_sim.in",
    grid_file=grid_file,
)
input_deck

In [ ]:
# Step 5 (optional): Run nullspace simulation command
# Replace SIM_CMD with your actual command.
import subprocess

SIM_CMD = None
# Example:
# SIM_CMD = "nsesSolve rf_sim.in"

if SIM_CMD:
    subprocess.run(SIM_CMD, shell=True, check=True)
    print("Simulation finished.")
else:
    print("Set SIM_CMD to your nullspace command, then re-run this cell.")

In [ ]:
# Step 6: Post-process result file into clean output HDF5
from post_processing import post_process

output_h5 = post_process(
    result_filename="rf_sim.in.h5",
    output_filename="rf_sim_out.h5",
    gridID=0,
    verbose=True,
)
output_h5

In [ ]:
# Quick output checks
import h5py

with h5py.File("rf_sim_out.h5", "r") as f:
    print("Datasets in rf_sim_out.h5:")
    print(list(f.keys()))